# The default model, next to the golden

`run(params)` is the model the docstrings describe. `run(params, match_golden=True)` copies three summer3 bugs so it can match `tweak_cases_calib`. Same initial population, same Vietnam data, same parameters, same solver. The only change is those three corrections, applied together on the default path. This is not a one-at-a-time decomposition, and it is not a recalibration: the parameters were chosen under the bugs, so the default epidemic is a different epidemic from the golden.

Use the **vacf (pixi)** kernel, as in [the inspect notebook](01-InspectRun.ipynb).


## What the golden got wrong

**Replacement deaths do not all become births at age 0.** Background deaths, TB deaths and deaths on treatment are declared as flows into `(mtb_naive, age 0)`. summer3's many-to-one scatter stores that compartment, then integrates with destination indices `0, 1, 2, …` instead. Edge `i` lands in compartment `i` of the map. Some of those compartments are `mtb_naive` at older ages. The rest are `incipient`. A death in the oldest `recovered` group is added to `incipient` at age 3. The default run sends every one of those deaths to `mtb_naive` at age 0.

**Progression is split twice.** The parameter `progression_prop_infectious` is meant to send a fraction `p` of progressions into high infectiousness and the rest into low, so the two weights sum to 1. At the default, `p` is 0.2. summer3 has already put weight `1/2` on each infectious stratum, and then multiplies by `[1 - p, p]`. The golden therefore progresses at about half the named rate (`0.4` and `0.1`). The default run uses `0.8` and `0.2`.

**`rel_infectiousness_subclin` multiplies clinical disease.** The clinical strata are ordered `clin`, then `subclin`. The golden wraps `[rel_infectiousness_subclin, 1]` onto that order, so clinical disease carries the subclinical weight and subclinical disease is fully infectious. The default run puts the weight on `subclin`.

Detail and the edge tables are in `reference/summer3-bugs.md`.


## Both runs

The solid lines in the plots are the golden-matching port (`match_golden=True`). The dashed lines are the default corrected model. The first figure is the seven disease states. The corrected seed already produces more active disease in the 1830s, and the gap does not close. By 2000 the default run has about six times as much active disease, and about six times the prevalence (`active` plus `treatment`). The never-infected population is smaller, because the extra progression and the corrected infectiousness have moved people out of `mtb_naive`.


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go

from vacf.constants import DISEASE_STATES, PARITY_ATOL, PARITY_RTOL
from vacf.model import calendar_years, run
from vacf.outputs import age_incidence, age_prevalence, compartments_by_label, disease_state_totals
from vacf.parameters import BASE_PARAMS

years = np.asarray(calendar_years())
matched = run(BASE_PARAMS, match_golden=True)
default = run(BASE_PARAMS)
matched_totals = {state: np.asarray(values) for state, values in disease_state_totals(matched).items()}
default_totals = {state: np.asarray(values) for state, values in disease_state_totals(default).items()}


In [ ]:
figure = go.Figure()
for state in DISEASE_STATES:
    figure.add_trace(go.Scatter(x=years, y=matched_totals[state], name=f"{state}, golden", mode="lines"))
    figure.add_trace(
        go.Scatter(
            x=years,
            y=default_totals[state],
            name=f"{state}, default",
            mode="lines",
            line={"dash": "dash"},
        )
    )
figure.update_layout(
    title="Disease states: golden-matching run and default run",
    xaxis_title="Year",
    yaxis_title="People",
    template="plotly_white",
    height=560,
)
figure.show()


## Incidence and prevalence

Incidence is progression into active disease, summed over age. Prevalence is everyone in `active` or `treatment`. Both are counts, not proportions. The default incidence in 2000 is about six times the golden, in line with the prevalence ratio. That is the doubled progression rate acting over two centuries, together with the other two corrections, not a change in the contact matrix or the birth series.


In [ ]:
def _summed(by_age: dict[str, np.ndarray]) -> np.ndarray:
    return sum(np.asarray(values) for values in by_age.values())


matched_incidence = _summed(age_incidence(matched))
default_incidence = _summed(age_incidence(default))
matched_prevalence = _summed(age_prevalence(matched))
default_prevalence = _summed(age_prevalence(default))

flows = go.Figure()
flows.add_trace(go.Scatter(x=years, y=matched_prevalence, name="Prevalence, golden", mode="lines"))
flows.add_trace(
    go.Scatter(x=years, y=default_prevalence, name="Prevalence, default", mode="lines", line={"dash": "dash"})
)
flows.add_trace(go.Scatter(x=years, y=matched_incidence, name="Incidence, golden", mode="lines"))
flows.add_trace(
    go.Scatter(x=years, y=default_incidence, name="Incidence, default", mode="lines", line={"dash": "dash"})
)
flows.update_layout(
    title="Prevalence and incidence under the two implementations",
    xaxis_title="Year",
    yaxis_title="People, or people per year",
    template="plotly_white",
    height=520,
)
flows.show()


## How far apart they are

The table is the default count divided by the golden-matching count, at four years. A value of 1 would mean the bugs did not matter. Active disease and treatment move together, because detection is the same function of clinical disease; both are several times the golden by 1900. `mtb_naive` falls below 1. Cleared infection does not move in the same direction at every year: it is higher just after the seed and lower by 1900, once the faster progression has drained the latent pool into active disease.

The assertions check two claims. The golden-matching run still agrees with the summer3 parquet. The default run does not: active disease in 2000 is more than twice the golden.


In [ ]:
marks = (1830, 1900, 2000, 2100)
columns = []
for year in marks:
    column = []
    for state in DISEASE_STATES:
        index = int(year - 1800)
        golden = float(matched_totals[state][index])
        corrected = float(default_totals[state][index])
        column.append("—" if golden == 0.0 else f"{corrected / golden:.2f}")
    columns.append(column)

table = go.Figure(
    data=[
        go.Table(
            header={"values": ["Disease state", *[str(year) for year in marks]]},
            cells={"values": [list(DISEASE_STATES), *columns]},
        )
    ]
)
table.update_layout(title="Default divided by golden-matching count", height=420)
table.show()

root = Path.cwd()
if not (root / "tests" / "golden" / "base").is_dir():
    root = root.parent
parquet = pd.read_parquet(root / "tests" / "golden" / "base" / "compartments.parquet")
labelled = compartments_by_label(matched)
for column, values in labelled.items():
    np.testing.assert_allclose(
        np.asarray(values),
        parquet[column].to_numpy(),
        rtol=PARITY_RTOL,
        atol=PARITY_ATOL,
    )

year_2000 = 200
assert float(default_totals["active"][year_2000]) > 2.0 * float(matched_totals["active"][year_2000])
ratio = float(default_totals["active"][year_2000] / matched_totals["active"][year_2000])
print(f"In 2000 the default model has {ratio:.1f} times as much active disease as the golden.")
